# Notebook 3: Predict Tree Trunk Count on New Data

This notebook applies the trained model (from Notebook 2) to a new, unseen shapefile of tree crown polygons. It:

1. Loads the new shapefile
2. Extracts the same morphological features used during training
3. Predicts trunk count for each polygon
4. Saves the results as a new shapefile with predictions

**Required inputs:**
- `model_config.json` -- feature list and metadata (from Notebook 2)
- `final_model_ridge.joblib` and/or `final_model_catboost.joblib` -- trained models (from Notebook 2)
- A new shapefile of tree crown polygons to predict on

In [ ]:
%pip install numpy pandas geopandas shapely matplotlib joblib

## 3.1 Configuration

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import math
import json
import joblib
import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION -- Update these paths for your data
# ============================================================
NEW_SHAPEFILE = "path/to/your/new_crowns.shp"  # <-- UPDATE THIS
OUTPUT_SHAPEFILE = "predicted_trunk_counts.shp"
MODEL_FILE = "final_model_ridge.joblib"         # or "final_model_catboost.joblib"
CONFIG_FILE = "model_config.json"
# ============================================================

# Load model config
with open(CONFIG_FILE) as f:
    config = json.load(f)

FEATURES = config['features']
TARGET_CRS = config['target_crs']

print(f"Model type: {config['best_model_type']}")
print(f"Features ({config['n_features']}): {FEATURES}")
print(f"Training performance: R2={config['test_r2']:.3f}, MAE={config['test_mae']:.2f}")
print(f"Target CRS: EPSG:{TARGET_CRS}")

## 3.2 Load and Prepare New Data

In [ ]:
# Load new shapefile
gdf = gpd.read_file(NEW_SHAPEFILE)
original_crs = gdf.crs
print(f"Loaded {len(gdf)} polygons from {NEW_SHAPEFILE}")
print(f"Original CRS: {original_crs}")

# Reproject to the training CRS (must match for correct metric units)
gdf = gdf.to_crs(epsg=TARGET_CRS)

# Explode multipart geometries
gdf = gdf.explode(index_parts=False).reset_index(drop=True)
gdf = gdf[gdf.geometry.type == 'Polygon'].copy()

# Remove contained polygons
sindex = gdf.sindex
to_drop = set()
for idx, row in gdf.iterrows():
    if idx in to_drop:
        continue
    candidates = list(sindex.query(row.geometry, predicate='contains'))
    for c in candidates:
        if c != idx and c not in to_drop:
            if row.geometry.contains(gdf.iloc[c].geometry):
                to_drop.add(c)
gdf = gdf.drop(index=list(to_drop)).reset_index(drop=True)

print(f"After cleaning: {len(gdf)} polygons")

## 3.3 Extract Features

The same feature extraction functions from Notebook 1 are used here to ensure consistency with the training data.

In [ ]:
def extract_features(gdf, feature_list):
    """
    Extract morphological features from polygons.
    Supports both 5-feature and 20-feature sets based on feature_list.
    """
    df = gdf.copy()

    # --- Basic metrics (always needed) ---
    df['perimeter'] = df.geometry.length
    df['area'] = df.geometry.area
    df['compactness'] = (4 * math.pi * df['area']) / (df['perimeter'] ** 2)
    df['perimeter_to_area'] = df['perimeter'] / df['area']

    # --- MRR axes ---
    def mrr_axes(geom):
        mrr = geom.minimum_rotated_rectangle
        coords = list(mrr.exterior.coords)
        side1 = math.hypot(coords[1][0] - coords[0][0], coords[1][1] - coords[0][1])
        side2 = math.hypot(coords[2][0] - coords[1][0], coords[2][1] - coords[1][1])
        return max(side1, side2), min(side1, side2)

    axes = df.geometry.apply(lambda g: pd.Series(mrr_axes(g), index=['major_axis', 'minor_axis']))
    df = pd.concat([df, axes], axis=1)

    # --- Eccentricity ---
    def eccentricity(row):
        major, minor = row['major_axis'], row['minor_axis']
        if major == 0 or major == minor:
            return 0.0
        if minor > major:
            major, minor = minor, major
        return math.sqrt(np.clip(1 - (minor ** 2 / major ** 2), 0, 1))
    df['eccentricity'] = df.apply(eccentricity, axis=1)

    # --- Extended features (only if needed) ---
    if len(feature_list) > 5:
        df['convexity'] = df.geometry.apply(lambda g: g.area / g.convex_hull.area if g.convex_hull.area > 0 else 1.0)
        df['aspect_ratio'] = df['major_axis'] / df['minor_axis'].replace(0, np.nan)
        df['aspect_ratio'] = df['aspect_ratio'].fillna(0)
        df['mrr_area_ratio'] = df['area'] / (df['major_axis'] * df['minor_axis']).replace(0, np.nan)
        df['mrr_area_ratio'] = df['mrr_area_ratio'].fillna(0)
        df['n_vertices'] = df.geometry.apply(lambda g: len(g.exterior.coords) - 1)
        df['boundary_sinuosity'] = df.geometry.apply(lambda g: g.length / g.convex_hull.length if g.convex_hull.length > 0 else 1.0)

        def count_concavities(geom):
            diff = geom.convex_hull.difference(geom)
            if diff.is_empty: return 0
            if diff.geom_type == 'Polygon': return 1
            if diff.geom_type in ('MultiPolygon', 'GeometryCollection'):
                return sum(1 for g in diff.geoms if g.geom_type == 'Polygon' and g.area > 0.01)
            return 0
        df['n_concavities'] = df.geometry.apply(count_concavities)

        def radial_stats(geom):
            centroid = geom.centroid
            coords = list(geom.exterior.coords)[:-1]
            dists = np.array([math.hypot(c[0] - centroid.x, c[1] - centroid.y) for c in coords])
            if len(dists) == 0:
                return 0, 0, 0, 1
            return dists.mean(), dists.std(), (dists.std() / dists.mean() if dists.mean() > 0 else 0), (dists.min() / dists.max() if dists.max() > 0 else 1)
        radial = df.geometry.apply(lambda g: pd.Series(radial_stats(g), index=['mean_radius', 'radius_std', 'radius_cv', 'radius_ratio']))
        df = pd.concat([df, radial], axis=1)

        df['equivalent_diameter'] = 2 * np.sqrt(df['area'] / math.pi)
        df['convex_hull_deficit'] = df.geometry.apply(lambda g: g.convex_hull.area - g.area)

        def l_ratio(row):
            if row['minor_axis'] == 0 or row['compactness'] == 0: return 0.0
            return (min(row['major_axis'], row['minor_axis']) / max(row['major_axis'], row['minor_axis'])) / (row['compactness'] ** 2)
        df['l_ratio'] = df.apply(l_ratio, axis=1)

    return df

# Extract features
print(f"Extracting {len(FEATURES)} features from {len(gdf)} polygons...")
gdf_feat = extract_features(gdf, FEATURES)

# Verify all required features are present
missing = [f for f in FEATURES if f not in gdf_feat.columns]
if missing:
    raise ValueError(f"Missing features: {missing}")

print("Feature extraction complete.")
gdf_feat[FEATURES].describe()

## 3.4 Load Model and Predict

In [ ]:
# Load trained model
model = joblib.load(MODEL_FILE)
print(f"Loaded model from {MODEL_FILE}")
print(f"Model type: {type(model).__name__}")

# Generate predictions
X_new = gdf_feat[FEATURES]
raw_predictions = model.predict(X_new)

# Post-process: round to integers, enforce minimum of 1
predictions = np.clip(np.round(raw_predictions), 1, None).astype(int)

# Attach predictions to the GeoDataFrame
gdf_feat['predicted_trunks'] = predictions
gdf_feat['raw_prediction'] = raw_predictions

print(f"\nPrediction summary:")
print(f"  Total polygons: {len(predictions)}")
print(f"  Predicted trunk range: {predictions.min()} - {predictions.max()}")
print(f"  Mean predicted trunks: {predictions.mean():.1f}")
print(f"  Median predicted trunks: {np.median(predictions):.0f}")
print(f"\nPredicted trunk count distribution:")
print(pd.Series(predictions).value_counts().sort_index())

## 3.5 Visualize Predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Prediction distribution histogram
ax = axes[0]
ax.hist(predictions, bins=range(1, predictions.max() + 2), edgecolor='black', alpha=0.7, align='left')
ax.set_xlabel('Predicted Trunk Count')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Predicted Trunk Counts')
ax.axvline(np.mean(predictions), color='red', linestyle='--', label=f'Mean={np.mean(predictions):.1f}')
ax.axvline(np.median(predictions), color='orange', linestyle='--', label=f'Median={np.median(predictions):.0f}')
ax.legend()

# 2. Prediction vs polygon area
ax = axes[1]
ax.scatter(gdf_feat['area'], predictions, alpha=0.4, s=20)
ax.set_xlabel('Polygon Area (m²)')
ax.set_ylabel('Predicted Trunk Count')
ax.set_title('Predicted Trunks vs Polygon Area')

# 3. Prediction vs compactness
ax = axes[2]
ax.scatter(gdf_feat['compactness'], predictions, alpha=0.4, s=20, color='green')
ax.set_xlabel('Compactness')
ax.set_ylabel('Predicted Trunk Count')
ax.set_title('Predicted Trunks vs Compactness')

fig.suptitle('Prediction Overview', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Spatial map of predictions (colored by predicted trunk count)
fig, ax = plt.subplots(1, 1, figsize=(12, 10))
gdf_feat.plot(
    column='predicted_trunks', ax=ax, legend=True,
    legend_kwds={'label': 'Predicted Trunk Count', 'shrink': 0.6},
    cmap='YlOrRd', edgecolor='black', linewidth=0.3, alpha=0.8
)
ax.set_title('Spatial Distribution of Predicted Trunk Counts')
ax.set_xlabel('Easting (m)')
ax.set_ylabel('Northing (m)')
ax.ticklabel_format(style='plain')
fig.tight_layout()
plt.show()

## 3.6 Flag Low-Confidence Predictions

Predictions far from the training data distribution may be less reliable. We flag polygons whose features fall outside the training range as potentially requiring manual review.

In [ ]:
# Flag predictions that may be less reliable:
# 1. Very high predictions (beyond training range -- model trained on 2-44)
# 2. Very small or very large polygons (extrapolation risk)

gdf_feat['confidence'] = 'normal'

# Flag high predictions (above training max of ~44)
gdf_feat.loc[gdf_feat['predicted_trunks'] > 30, 'confidence'] = 'low - high count'

# Flag predictions where raw value differs significantly from rounded
gdf_feat.loc[
    (gdf_feat['raw_prediction'] - gdf_feat['predicted_trunks']).abs() > 0.4,
    'confidence'
] = 'borderline'

confidence_counts = gdf_feat['confidence'].value_counts()
print("Prediction confidence flags:")
print(confidence_counts.to_string())
print(f"\n{confidence_counts.get('normal', 0)} / {len(gdf_feat)} predictions flagged as normal confidence")

## 3.7 Save Output Shapefile

In [ ]:
# Prepare output GeoDataFrame
output_cols = ['predicted_trunks', 'confidence', 'geometry']

# Include original columns from the input shapefile (if any)
original_cols = [c for c in gdf.columns if c not in gdf_feat.columns and c != 'geometry']
output_cols = original_cols + output_cols

# Also include key features for reference
feature_cols = [f for f in FEATURES if f in gdf_feat.columns]
output_cols = output_cols[:1] + feature_cols + output_cols[len(original_cols):]

# Shapefile column name limit (10 chars) -- abbreviate
col_rename = {
    'predicted_trunks': 'pred_trunk',
    'confidence': 'confidence',
    'perimeter': 'perimeter',
    'area': 'area',
    'compactness': 'compact',
    'perimeter_to_area': 'p_to_a',
    'eccentricity': 'eccentric',
}

out_gdf = gdf_feat[list(set(output_cols))].copy()
out_gdf = out_gdf.rename(columns=col_rename)

# Reproject back to the original CRS
out_gdf = gpd.GeoDataFrame(out_gdf, geometry='geometry', crs=f'EPSG:{TARGET_CRS}')
if original_crs is not None:
    out_gdf = out_gdf.to_crs(original_crs)

# Save
out_gdf.to_file(OUTPUT_SHAPEFILE)
print(f"Saved predictions to: {OUTPUT_SHAPEFILE}")
print(f"  Polygons: {len(out_gdf)}")
print(f"  Columns: {list(out_gdf.columns)}")
print(f"  CRS: {out_gdf.crs}")
print(f"\nDone! Open {OUTPUT_SHAPEFILE} in QGIS/ArcGIS to visualize results.")